In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: False
GPU count: 0


In [3]:
import json
import os
from collections import Counter
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
ACTS_FILE = "Phase_1_Pretraining/Train/train_acts.json"

In [4]:
def stream_json_array(filepath):
    decoder = json.JSONDecoder()

    with open(filepath, "r", encoding="utf-8") as f:
        buffer = ""
        pos = 0
        started = False

        while True:
            chunk = f.read(1024 * 1024) 
            if not chunk:
                break

            buffer += chunk

            while True:
                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1

                if not started:
                    if pos >= len(buffer):
                        break

                    if buffer[pos] == "[":
                        pos += 1
                        started = True
                    else:
                        raise ValueError("Expected JSON array")

                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1

                # End of JSON array
                if pos < len(buffer) and buffer[pos] == "]":
                    return

                try:
                    obj, end = decoder.raw_decode(buffer, pos)
                except json.JSONDecodeError:
                    # Need more data
                    break

                yield obj
                pos = end

                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1

                if pos < len(buffer) and buffer[pos] == ",":
                    pos += 1
                elif pos < len(buffer) and buffer[pos] == "]":
                    return

            # Keep only the unprocessed part
            buffer = buffer[pos:]
            pos = 0

In [5]:
acts_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=ACTS_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

In [8]:
import re
import time
import heapq
import pickle
from collections import Counter, defaultdict

# ---- settings -------------------------------------------------------------
VOCAB_SIZE     = 25_000                # FINAL size: 256 bytes + merges + special tokens
LOWERCASE      = True                 # your old pipeline lowercased; set True to keep that
SPECIAL_TOKENS = ["<|endoftext|>"]
TARGET_WORDS   = 200_000_000            # how much text to learn the BPE from
SAVE_PATH      = "legal_bpe_v2.pkl"

# GPT-2 style pre-tokenizer: the space is attached to the *next* word  ->  "dog", " ate", " cat"
# (whitespace is KEPT, and the exact same regex is used for training and for encoding)
GPT2_PATTERN = (
    r"'s|'t|'re|'ve|'m|'ll|'d"
    r"| ?[^\W\d_]+"        # optional space + letters
    r"| ?\d+"               # optional space + digits
    r"| ?[^\s\w]+"         # optional space + punctuation / symbols
    r"|\s+(?!\S)"          # whitespace run not followed by a non-space
    r"|\s+"                 # remaining whitespace (newlines etc.)
)
PATTERN = re.compile(GPT2_PATTERN)

In [9]:
def text_iterator(max_words=TARGET_WORDS):
    seen = 0
    for record in stream_json_array(acts_path):      # from your cells 3-4
        text = record["text"]
        seen += len(text.split())
        yield text
        if seen >= max_words:
            break

t0 = time.time()
word_freq = Counter()                                # bytes of one pre-token -> count
for text in text_iterator():
    if LOWERCASE:
        text = text.lower()
    for piece in PATTERN.findall(text):
        word_freq[piece.encode("utf-8")] += 1

print(f"Total pre-tokens : {sum(word_freq.values()):,}")
print(f"Unique pre-tokens: {len(word_freq):,}")
print(f"Time             : {time.time() - t0:.1f}s")
print("Most common      :", [(w.decode('utf-8', 'replace'), c) for w, c in word_freq.most_common(8)])

Total pre-tokens : 150,270,943
Unique pre-tokens: 377,041
Time             : 89.7s
Most common      : [(' the', 9583960), (' of', 6657417), ('.', 5166418), (',', 5159466), ('\n', 4033621), (')', 3138420), (' to', 3010278), (' and', 2739498)]


In [10]:
def train_bpe(word_freq, vocab_size, num_special, log_every=500):
    n_merges = vocab_size - 256 - num_special
    assert n_merges > 0, "VOCAB_SIZE too small"
    t0 = time.time()

    words = [list(w) for w in word_freq]             # each word = list of token ids (starts as bytes)
    freqs = list(word_freq.values())

    # pair counts + inverted index (pair -> set of word indices that contain it)
    pair_counts = defaultdict(int)
    where = defaultdict(set)
    for wi, w in enumerate(words):
        f = freqs[wi]
        for pair in zip(w, w[1:]):
            pair_counts[pair] += f
            where[pair].add(wi)

    heap = [(-c, p) for p, c in pair_counts.items()]  # lazy max-heap (stale entries skipped on pop)
    heapq.heapify(heap)

    merges = []                                      # [((a, b), new_id), ...]  same format as your old pkl
    while len(merges) < n_merges and heap:
        neg_c, pair = heapq.heappop(heap)
        c = pair_counts.get(pair, 0)
        if c <= 0 or -neg_c != c:
            continue                                 # stale heap entry

        new_id = 256 + len(merges)
        merges.append((pair, new_id))
        a, b = pair
        touched = set()

        for wi in list(where[pair]):
            w = words[wi]
            f = freqs[wi]
            if not any(w[i] == a and w[i + 1] == b for i in range(len(w) - 1)):
                continue                             # index is lazy; word no longer has the pair

            for p in zip(w, w[1:]):                  # remove old counts of this word
                pair_counts[p] -= f
                touched.add(p)

            new_w, i = [], 0                         # apply the merge
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                    new_w.append(new_id)
                    i += 2
                else:
                    new_w.append(w[i])
                    i += 1
            words[wi] = new_w

            for p in zip(new_w, new_w[1:]):          # add new counts of this word
                pair_counts[p] += f
                where[p].add(wi)
                touched.add(p)

        del where[pair]
        pair_counts.pop(pair, None)
        touched.discard(pair)
        for p in touched:
            if pair_counts[p] > 0:
                heapq.heappush(heap, (-pair_counts[p], p))

        if len(merges) % log_every == 0:
            print(f"Merges: {len(merges):,}/{n_merges:,} | Pair: {pair} | "
                  f"Frequency: {c:,} | {time.time() - t0:.0f}s")

    print(f"Done: {len(merges):,} merges in {time.time() - t0:.1f}s")
    return merges

merges = train_bpe(word_freq, VOCAB_SIZE, len(SPECIAL_TOKENS))

Merges: 500/24,743 | Pair: (117, 334) | Frequency: 124,703 | 21s
Merges: 1,000/24,743 | Pair: (502, 327) | Frequency: 54,174 | 24s
Merges: 1,500/24,743 | Pair: (101, 493) | Frequency: 31,482 | 25s
Merges: 2,000/24,743 | Pair: (1936, 426) | Frequency: 20,764 | 27s
Merges: 2,500/24,743 | Pair: (547, 324) | Frequency: 14,422 | 28s
Merges: 3,000/24,743 | Pair: (266, 273) | Frequency: 10,650 | 29s
Merges: 3,500/24,743 | Pair: (103, 299) | Frequency: 8,123 | 30s
Merges: 4,000/24,743 | Pair: (2442, 115) | Frequency: 6,454 | 31s
Merges: 4,500/24,743 | Pair: (4379, 433) | Frequency: 5,180 | 31s
Merges: 5,000/24,743 | Pair: (4537, 746) | Frequency: 4,205 | 32s
Merges: 5,500/24,743 | Pair: (1558, 409) | Frequency: 3,526 | 33s
Merges: 6,000/24,743 | Pair: (779, 293) | Frequency: 2,975 | 34s
Merges: 6,500/24,743 | Pair: (3271, 386) | Frequency: 2,495 | 34s
Merges: 7,000/24,743 | Pair: (4421, 1280) | Frequency: 2,139 | 34s
Merges: 7,500/24,743 | Pair: (1614, 283) | Frequency: 1,872 | 35s
Merges: 8,0

In [11]:
bpe_model = {
    "vocab_size": 256 + len(merges) + len(SPECIAL_TOKENS),
    "merges": merges,                      # [((a, b), new_id), ...]
    "special_tokens": SPECIAL_TOKENS,      # ids are assigned right after the last merge
    "lowercase": LOWERCASE,
    "pattern": GPT2_PATTERN,
}

with open(SAVE_PATH, "wb") as f:
    pickle.dump(bpe_model, f)

print("BPE tokenizer saved ->", SAVE_PATH)
print("Vocabulary size:", bpe_model["vocab_size"])
print("Number of merges:", len(merges))

BPE tokenizer saved -> legal_bpe_v2.pkl
Vocabulary size: 25000
Number of merges: 24743


In [12]:
token_bytes = {i: bytes([i]) for i in range(256)}
for (l, r), new_id in merges:
    token_bytes[new_id] = token_bytes[l] + token_bytes[r]

for (l, r), new_id in merges[:5] + merges[5000:5005] + merges[-5:]:
    print(new_id, repr(token_bytes[new_id].decode("utf-8", "replace")))

256 ' t'
257 ' a'
258 ' o'
259 'he'
260 ' the'
5256 ' quarry'
5257 ' soil'
5258 ' concerning'
5259 ' cell'
5260 ' resu'
24994 ' tiruppur'
24995 ' polymers'
24996 'volatile'
24997 ' pint'
24998 ' trouble'


In [9]:
import os, json, random, time, re, heapq, pickle, math
from collections import Counter, defaultdict
import numpy as np
from huggingface_hub import hf_hub_download, HfApi

REPO = "Exploration-Lab/CS-781-Capstone"

# ---------------- what to build ----------------
TOTAL_TOKENS = 150_000_000          # train + val together
VAL_TOKENS = 5_000_000
RATIOS = {                          # share of the TOTAL_TOKENS taken from each source (must sum to 1)
    "acts": 0.05,
    "high_courts": 0.4,
    "supreme_court": 0.2,
    "tribunals": 0.35,
}
assert abs(sum(RATIOS.values()) - 1.0) < 1e-9

# ---------------- tokenizer ----------------
VOCAB_SIZE = 30_000                 # final size: 256 bytes + merges + special tokens
LOWERCASE = False                   # keep the original casing
SPECIAL_TOKENS = ["<|endoftext|>"]
TOK_TRAIN_WORDS = 30_000_000        # words used to learn the BPE merges (split by RATIOS)

# ---------------- collection ----------------
TOK_PER_WORD = 1.35                 # rough guess, only used to decide how much text to collect
SAFETY = 1.25                       # collect 25% extra text so no source runs short
MIN_DOC_WORDS = 50
SEED = 42

TMP_ROOT = "/kaggle/temp"
try:
    os.makedirs(TMP_ROOT, exist_ok=True)
except Exception:
    TMP_ROOT = "/tmp"
RAW_DIR = f"{TMP_ROOT}/raw"
MIX_DIR = f"{TMP_ROOT}/mix"
OUT_DIR = "/kaggle/working"
for d in (RAW_DIR, MIX_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# ---------------- find the four training files in the HF repo ----------------
api = HfApi()
train_files = [f for f in api.list_repo_files(REPO, repo_type="dataset")
               if "Phase_1" in f and "Train" in f and f.endswith(".json")]
SOURCE_FILES = {}
for name in RATIOS:
    match = [f for f in train_files if name in os.path.basename(f)]
    assert len(match) == 1, f"could not find exactly one file for '{name}': {match}"
    SOURCE_FILES[name] = match[0]
print(SOURCE_FILES)


# ---------------- helpers ----------------
if "stream_json_array" not in globals():          # use your own reader if it already exists
    def stream_json_array(path, chunk_size=1 << 24):
        dec = json.JSONDecoder()
        ws = " \t\r\n,"
        with open(path, "r", encoding="utf-8") as f:
            buf = ""
            while "[" not in buf:
                chunk = f.read(chunk_size)
                if not chunk:
                    return
                buf += chunk
            pos = buf.index("[") + 1
            while True:
                while pos < len(buf) and buf[pos] in ws:
                    pos += 1
                if pos >= len(buf):
                    chunk = f.read(chunk_size)
                    if not chunk:
                        return
                    buf, pos = chunk, 0
                    continue
                if buf[pos] == "]":
                    return
                try:
                    obj, pos = dec.raw_decode(buf, pos)
                    yield obj
                except json.JSONDecodeError:
                    chunk = f.read(chunk_size)
                    if not chunk:
                        raise
                    buf, pos = buf[pos:] + chunk, 0


def get_text(rec):
    if isinstance(rec, str):
        return rec
    if "text" in rec:
        return rec["text"]
    strings = [v for v in rec.values() if isinstance(v, str)]
    return max(strings, key=len) if strings else ""


def collect_source(name, raw_path, word_quota, out_path, seed):
    """Stream the whole file once, keep every `stride`-th document (spread over the entire file,
    not just the beginning) until `word_quota` words are collected. Output: one JSON line per doc."""
    rng = random.Random(seed)
    size = os.path.getsize(raw_path)

    n = total_bytes = total_words = 0                       # estimate document size from the first 300 docs
    for rec in stream_json_array(raw_path):
        t = get_text(rec)
        n += 1
        total_bytes += len(t.encode("utf-8"))
        total_words += len(t.split())
        if n >= 300:
            break
    avg_bytes, avg_words = total_bytes / n, max(total_words / n, 1)
    est_docs = size / (avg_bytes * 1.02)
    docs_needed = word_quota / avg_words
    stride = max(1, int(0.85 * est_docs / docs_needed))     # 0.85: the estimate is slightly optimistic
    offset = rng.randrange(stride)

    words = kept = 0
    with open(out_path, "w", encoding="utf-8") as out:
        for idx, rec in enumerate(stream_json_array(raw_path)):
            if idx % stride != offset:
                continue
            t = get_text(rec)
            w = len(t.split())
            if w < MIN_DOC_WORDS:
                continue
            out.write(json.dumps({"t": t}, ensure_ascii=False) + "\n")
            words += w
            kept += 1
            if words >= word_quota:
                break
    print(f"  {name}: kept {kept:,} docs, {words:,} words (quota {word_quota:,}), stride {stride}")
    return words, kept

{'acts': 'Phase_1_Pretraining/Train/train_acts.json', 'high_courts': 'Phase_1_Pretraining/Train/train_high_courts.json', 'supreme_court': 'Phase_1_Pretraining/Train/train_supreme_court.json', 'tribunals': 'Phase_1_Pretraining/Train/train_tribunals.json'}


In [10]:
collected = {}
for name, fn in SOURCE_FILES.items():
    t0 = time.time()
    word_quota = int(RATIOS[name] * TOTAL_TOKENS / TOK_PER_WORD * SAFETY)
    print(f"[{name}] downloading {fn} ...")
    raw_path = hf_hub_download(REPO, fn, repo_type="dataset", local_dir=RAW_DIR)
    print(f"  raw file: {os.path.getsize(raw_path) / 1e9:.2f} GB")
    collected[name] = collect_source(name, raw_path, word_quota, f"{MIX_DIR}/{name}.jsonl", SEED)
    os.remove(raw_path)                                    # free the disk before the next source
    print(f"  done in {(time.time() - t0) / 60:.1f} min")

short = [n for n, (w, _) in collected.items() if w < RATIOS[n] * TOTAL_TOKENS / TOK_PER_WORD]
if short:
    print("WARNING: these sources are small and may not reach their share:", short)

[acts] downloading Phase_1_Pretraining/Train/train_acts.json ...


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

  raw file: 0.73 GB
  acts: kept 1,179 docs, 6,947,249 words (quota 6,944,444), stride 14
  done in 0.2 min
[high_courts] downloading Phase_1_Pretraining/Train/train_high_courts.json ...


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

  raw file: 10.67 GB
  high_courts: kept 84,658 docs, 55,555,945 words (quota 55,555,555), stride 25
  done in 2.4 min
[supreme_court] downloading Phase_1_Pretraining/Train/train_supreme_court.json ...


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

  raw file: 1.32 GB
  supreme_court: kept 7,090 docs, 27,780,912 words (quota 27,777,777), stride 6
  done in 0.3 min
[tribunals] downloading Phase_1_Pretraining/Train/train_tribunals.json ...


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

  raw file: 6.48 GB
  tribunals: kept 42,385 docs, 48,611,835 words (quota 48,611,111), stride 18
  done in 1.7 min


In [7]:
from huggingface_hub import hf_hub_download, HfApi

test_files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset") if "Phase_1" in f and "Test" in f]
test_data = json.load(open(hf_hub_download(REPO, test_files[0], repo_type="dataset"), encoding="utf-8"))
prompts = [v["text"] for v in test_data.values()]

print("=== first 300 characters of 3 test prompts ===")
for p in prompts[:3]:
    print(repr(p[:1000]), "\n")


def unigram_counts(path, max_words=3_000_000):
    c, n = Counter(), 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            w = json.loads(line)["t"].lower().split()
            c.update(w)
            n += len(w)
            if n >= max_words:
                break
    return c, n


SMOOTH_V = 2_000_000
print("=== how predictable are the test prompts from each source? (higher = more similar) ===")
for name in RATIOS:
    c, n = unigram_counts(f"{MIX_DIR}/{name}.jsonl")
    total, count = 0.0, 0
    for p in prompts:
        for w in p.lower().split():
            total += math.log((c.get(w, 0) + 1) / (n + SMOOTH_V))
            count += 1
    print(f"{name:>14}: mean log-prob per word = {total / count:.3f}")

=== first 300 characters of 3 test prompts ===
'The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage. The issue is no more res integra as the issue has been considered by this Court time and again. 6. When a person challenges Section 4 Notification on any ground, it should be challenged within a reasonable period, and if the acquisition is challenged at a belated stage, the petition deserves to be dismissed only on this count. (Vide Hari Singh & Ors. Vs. State of U.P., AIR 1984 SC 1020). 7. A Constitution Bench of this Court, in [Aflatoon & Ors. Vs. Lt. Governor, Delhi & Ors.] AIR 1974 SC 2077, while dealing with the issue, observed as under:- ".... to have sat on the fence and allowed the government to complete the acquisition on the basis that notification under Section 4 and the declaration under Section 6 were valid and then to attack the notification on the grounds which were available to them at the time when the 

In [11]:
GPT2_PATTERN = (
    r"'s|'t|'re|'ve|'m|'ll|'d"
    r"| ?[^\W\d_]+"          # optional space + letters
    r"| ?\d+"                # optional space + digits
    r"| ?[^\s\w]+"           # optional space + punctuation / symbols
    r"|\s+(?!\S)"            # whitespace run not followed by a non-space
    r"|\s+"                  # remaining whitespace (newlines etc.)
)
PATTERN = re.compile(GPT2_PATTERN)

# ---- 1) count pre-tokens on a sample that follows RATIOS ----
t0 = time.time()
word_freq = Counter()
for name, share in RATIOS.items():
    quota, seen = int(share * TOK_TRAIN_WORDS), 0
    with open(f"{MIX_DIR}/{name}.jsonl", encoding="utf-8") as f:
        for line in f:
            text = json.loads(line)["t"]
            if LOWERCASE:
                text = text.lower()
            seen += len(text.split())
            for piece in PATTERN.findall(text):
                word_freq[piece.encode("utf-8")] += 1
            if seen >= quota:
                break
    print(f"  {name}: {seen:,} words used for the tokenizer")

before = len(word_freq)
word_freq = Counter({w: c for w, c in word_freq.items() if c >= 2})   # drop one-off strings (names, typos): saves RAM/time
print(f"unique pre-tokens: {before:,} -> {len(word_freq):,} after dropping singletons | {time.time() - t0:.0f}s")


# ---- 2) fast incremental BPE training ----
def train_bpe(word_freq, vocab_size, num_special, log_every=2000):
    n_merges = vocab_size - 256 - num_special
    assert n_merges > 0, "VOCAB_SIZE too small"
    t0 = time.time()

    words = [list(w) for w in word_freq]
    freqs = list(word_freq.values())

    pair_counts = defaultdict(int)
    where = defaultdict(set)
    for wi, w in enumerate(words):
        f = freqs[wi]
        for pair in zip(w, w[1:]):
            pair_counts[pair] += f
            where[pair].add(wi)

    heap = [(-c, p) for p, c in pair_counts.items()]
    heapq.heapify(heap)

    merges = []
    while len(merges) < n_merges and heap:
        neg_c, pair = heapq.heappop(heap)
        c = pair_counts.get(pair, 0)
        if c <= 0 or -neg_c != c:
            continue

        new_id = 256 + len(merges)
        merges.append((pair, new_id))
        a, b = pair
        touched = set()

        for wi in list(where[pair]):
            w = words[wi]
            f = freqs[wi]
            if not any(w[i] == a and w[i + 1] == b for i in range(len(w) - 1)):
                continue

            for p in zip(w, w[1:]):
                pair_counts[p] -= f
                touched.add(p)

            new_w, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                    new_w.append(new_id)
                    i += 2
                else:
                    new_w.append(w[i])
                    i += 1
            words[wi] = new_w

            for p in zip(new_w, new_w[1:]):
                pair_counts[p] += f
                where[p].add(wi)
                touched.add(p)

        del where[pair]
        pair_counts.pop(pair, None)
        touched.discard(pair)
        for p in touched:
            if pair_counts[p] > 0:
                heapq.heappush(heap, (-pair_counts[p], p))

        if len(merges) % log_every == 0:
            print(f"Merges: {len(merges):,}/{n_merges:,} | Frequency: {c:,} | {time.time() - t0:.0f}s")

    print(f"Done: {len(merges):,} merges in {time.time() - t0:.1f}s")
    return merges


merges = train_bpe(word_freq, VOCAB_SIZE, len(SPECIAL_TOKENS))

bpe_model = {
    "vocab_size": 256 + len(merges) + len(SPECIAL_TOKENS),
    "merges": merges,
    "special_tokens": SPECIAL_TOKENS,
    "lowercase": LOWERCASE,
    "pattern": GPT2_PATTERN,
}
TOKENIZER_V3_PATH = f"{OUT_DIR}/legal_bpe_v3.pkl"
with open(TOKENIZER_V3_PATH, "wb") as f:
    pickle.dump(bpe_model, f)
print("saved", TOKENIZER_V3_PATH, "| vocab size:", bpe_model["vocab_size"])
del word_freq

  acts: 1,502,134 words used for the tokenizer
  high_courts: 12,056,934 words used for the tokenizer
  supreme_court: 6,010,445 words used for the tokenizer
  tribunals: 10,500,158 words used for the tokenizer
unique pre-tokens: 299,994 -> 150,699 after dropping singletons | 38s
Merges: 2,000/29,743 | Frequency: 5,838 | 10s
Merges: 4,000/29,743 | Frequency: 2,021 | 11s
Merges: 6,000/29,743 | Frequency: 1,019 | 12s
Merges: 8,000/29,743 | Frequency: 620 | 13s
Merges: 10,000/29,743 | Frequency: 417 | 13s
Merges: 12,000/29,743 | Frequency: 299 | 14s
Merges: 14,000/29,743 | Frequency: 225 | 14s
Merges: 16,000/29,743 | Frequency: 176 | 14s
Merges: 18,000/29,743 | Frequency: 142 | 15s
Merges: 20,000/29,743 | Frequency: 116 | 15s
Merges: 22,000/29,743 | Frequency: 96 | 16s
Merges: 24,000/29,743 | Frequency: 81 | 16s
Merges: 26,000/29,743 | Frequency: 71 | 16s
Merges: 28,000/29,743 | Frequency: 62 | 16s
Done: 29,743 merges in 16.1s
saved /kaggle/working/legal_bpe_v3.pkl | vocab size: 30000


In [12]:
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

from datasets import load_dataset


class LegalBPETokenizer:

    def __init__(self, model_path):

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])

        # pair -> rank (lower rank = learned earlier = merged first)
        self.merge_rank = {
            tuple(pair): rank
            for rank, (pair, _) in enumerate(self.merges)
        }

        # id -> bytes table (O(1) decode)
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]

        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")

        self._special_re = (
            re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
            if self.special_tokens else None
        )
        self._cache = {}

    def _encode_piece(self, piece_bytes):

        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached

        ids = list(piece_bytes)

        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break

            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):

        if self.lowercase:
            text = text.lower()

        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):

        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids):

        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])

        return b"".join(chunks).decode("utf-8", errors="replace")


In [13]:
# needs the LegalBPETokenizer class (the class cell you already have); load the NEW pkl
tokenizer = LegalBPETokenizer(TOKENIZER_V3_PATH)
assert tokenizer.vocab_size < 65535, "ids must fit in uint16"

TRAIN_OUT = f"{OUT_DIR}/train_v3.bin"
VAL_OUT = f"{OUT_DIR}/val_v3.bin"
VAL_FRAC = VAL_TOKENS / TOTAL_TOKENS

rng = random.Random(SEED)
need = {n: int(RATIOS[n] * TOTAL_TOKENS) for n in RATIOS}
got = {n: 0 for n in RATIOS}
words_seen = {n: 0 for n in RATIOS}
exhausted = {n: False for n in RATIOS}
files = {n: open(f"{MIX_DIR}/{n}.jsonl", encoding="utf-8") for n in RATIOS}


def estimate_doc_tokens(path, n_docs=100):                 # rough average document length in tokens
    words = docs = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            words += len(json.loads(line)["t"].split())
            docs += 1
            if docs >= n_docs:
                break
    return max(words / max(docs, 1) * TOK_PER_WORD, 1.0)


avg_doc_tokens = {n: estimate_doc_tokens(f"{MIX_DIR}/{n}.jsonl") for n in RATIOS}
full_tokens = {n: 0 for n in RATIOS}
docs_done = {n: 0 for n in RATIOS}
train_count = val_count = 0
last_print = 0
t0 = time.time()

with open(TRAIN_OUT, "wb") as train_f, open(VAL_OUT, "wb") as val_f:
    while True:
        active = [n for n in RATIOS if got[n] < need[n] and not exhausted[n]]
        if not active:
            break
        # pick the next document's source in proportion to the NUMBER of documents each source still needs
        # (remaining tokens / average doc length) => sources are spread evenly through the whole file
        weights = [(need[m] - got[m]) / avg_doc_tokens[m] for m in active]
        n = rng.choices(active, weights=weights)[0]
        line = files[n].readline()
        if not line:
            exhausted[n] = True
            continue
        text = json.loads(line)["t"]
        ids = tokenizer.encode(text, add_eos=True)
        full_tokens[n] += len(ids)
        docs_done[n] += 1
        if docs_done[n] >= 20:
            avg_doc_tokens[n] = full_tokens[n] / docs_done[n]
        ids = ids[:need[n] - got[n]]
        words_seen[n] += len(text.split())
        arr = np.asarray(ids, dtype=np.uint16)
        if rng.random() < VAL_FRAC:
            arr.tofile(val_f)
            val_count += len(ids)
        else:
            arr.tofile(train_f)
            train_count += len(ids)
        got[n] += len(ids)

        total = train_count + val_count
        if total - last_print >= 5_000_000:
            last_print = total
            print(f"{total:,} tokens | {total / (time.time() - t0):,.0f} tok/s | {(time.time() - t0) / 60:.1f} min")

for f in files.values():
    f.close()

print("\nFinished!")
print(f"train tokens: {train_count:,} | val tokens: {val_count:,} | total: {train_count + val_count:,}")
for n in RATIOS:
    tpw = got[n] / max(words_seen[n], 1)
    flag = "" if got[n] >= 0.99 * need[n] else "   <-- SHORT: raise SAFETY and re-run the collection cell for this source"
    print(f"{n:>14}: {got[n]:>12,} tokens ({got[n] / max(train_count + val_count, 1):.1%}) | {tpw:.2f} tokens/word{flag}")

5,013,297 tokens | 825,758 tok/s | 0.1 min
10,016,870 tokens | 837,700 tok/s | 0.2 min
15,017,341 tokens | 854,448 tok/s | 0.3 min
20,017,440 tokens | 866,159 tok/s | 0.4 min
25,017,915 tokens | 862,922 tok/s | 0.5 min
30,020,098 tokens | 865,776 tok/s | 0.6 min
35,039,084 tokens | 861,767 tok/s | 0.7 min
40,044,145 tokens | 874,215 tok/s | 0.8 min
45,048,343 tokens | 887,833 tok/s | 0.8 min
50,048,749 tokens | 894,179 tok/s | 0.9 min
55,051,855 tokens | 893,847 tok/s | 1.0 min
60,051,972 tokens | 898,637 tok/s | 1.1 min
65,053,284 tokens | 901,432 tok/s | 1.2 min
70,054,599 tokens | 904,241 tok/s | 1.3 min
75,061,517 tokens | 908,221 tok/s | 1.4 min
80,063,944 tokens | 911,286 tok/s | 1.5 min
85,068,110 tokens | 913,548 tok/s | 1.6 min
90,069,211 tokens | 914,349 tok/s | 1.6 min
95,071,008 tokens | 914,357 tok/s | 1.7 min
100,072,358 tokens | 916,254 tok/s | 1.8 min
105,232,240 tokens | 917,083 tok/s | 1.9 min
110,232,666 tokens | 918,294 tok/s | 2.0 min
115,234,530 tokens | 917,267 t